# DataTrust — Auditoría de calidad: tabla `reviews`

## Contexto

La tabla `reviews` contiene 1.033.523 reseñas de huéspedes sobre alojamientos de Airbnb Barcelona, con el texto del comentario, la fecha, y quién la escribió.

## Problema

Las reseñas son el mecanismo de confianza principal entre huéspedes y anfitriones en cualquier plataforma de alquiler turístico. Si ese mecanismo se puede manipular o contiene ruido (duplicados, huérfanas, texto repetido), la confianza que transmite es menor de lo que parece.

## Objetivo de este notebook

Auditar `reviews` en cinco dimensiones: completitud, duplicados, integridad referencial (con SQL, contra `listings`), patrones de contenido relacionados con autenticidad y riqueza informativa, y coherencia de fechas. No se corrige nada todavía, ni se afirma que un patrón sospechoso sea necesariamente una reseña falsa, eso requeriría una fuente externa que lo confirme.

In [1]:
import pandas as pd
import sqlite3

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

reviews = pd.read_csv("../data/airbnb_bcn/reviews.csv", low_memory=False)
listings = pd.read_csv("../data/airbnb_bcn/listings.csv", low_memory=False)
print("Filas reviews:", reviews.shape[0], "| Columnas:", reviews.shape[1])

Filas reviews: 1033523 | Columnas: 6


## 1. Completitud

**Criterio (mismo umbral objetivo que en el notebook de `listings`, para mantener consistencia entre notebooks)**: `listing_id`, `id`, `date` y `reviewer_id` son campos estructurales, deberían tener 0% de nulos. Para el resto de columnas: 100% de nulos → caso crítico; ≥10% y <100% → problema de completitud a documentar (umbral operativo definido para este proyecto, no un estándar externo); <10% → no se reporta como hallazgo relevante.

In [3]:
nulos = reviews.isna().sum()
nulos_pct = (nulos / len(reviews) * 100).round(4)
print(pd.DataFrame({"nulos": nulos, "pct": nulos_pct.astype(str) + "%"}))

               nulos      pct
listing_id         0     0.0%
id                 0     0.0%
date               0     0.0%
reviewer_id        0     0.0%
reviewer_name      2  0.0002%
comments         114   0.011%


**Interpretación**: los campos estructurales (`listing_id`, `id`, `date`, `reviewer_id`) están completos, con 0 nulos, tal como exige el criterio para esos campos. `comments` presenta 114 nulos (0,011%) y `reviewer_name` 2 nulos (0,0002%). Ambos porcentajes están muy por debajo del umbral operativo del 10% definido en el criterio, por lo que no se consideran hallazgos relevantes de completitud: el criterio se definió antes de ver el resultado y se aplica igual, no se ajusta después.

## 2. Duplicados

**Criterio**: no debería haber filas completas repetidas. Además, comprobamos si el mismo texto de comentario se repite exactamente en distintas reseñas, lo cual no es necesariamente un error pero sí una señal a vigilar.

In [4]:
print("Filas duplicadas completas:", reviews.duplicated().sum())
print("IDs de reseña unicos:", reviews["id"].is_unique)

comentarios = reviews["comments"].dropna()
print("Comentarios duplicados exactos (mismo texto, distinta reseña):", comentarios.duplicated().sum())

Filas duplicadas completas: 0
IDs de reseña unicos: True


Comentarios duplicados exactos (mismo texto, distinta reseña): 48302


**Interpretación**: no se detectan filas completamente duplicadas y el identificador de reseña es único. Sin embargo, **48.302 registros de `comments` presentan un texto que ya había aparecido antes en otra reseña** (nota: `.duplicated()` cuenta apariciones posteriores a la primera de cada texto repetido, no el total de filas que comparten ese texto). Este resultado no demuestra por sí solo un problema de autenticidad: puede deberse tanto a frases cortas y comunes que distintos huéspedes escriben de forma independiente ("Great!", "Muy buena ubicación") como a una reutilización real de texto. Se investiga con más contexto en la Sección 4.

## 3. Integridad referencial (SQL)

**Criterio**: toda reseña debería pertenecer a un alojamiento que existe en `listings`. Se comprueba con SQL (SQLite), un `LEFT JOIN` es la forma natural de detectar registros huérfanos, más que replicar la misma lógica en pandas.

**Nota técnica**: la base de datos completa (las 3 tablas enteras) ya está construida en `database/airbnb.db` por `database/construir_base_datos.py`. En esta sección solo se establece la conexión y se realizan las consultas SQL; no se vuelve a cargar ningún CSV.

In [5]:
con = sqlite3.connect("../database/airbnb.db")

query = """
SELECT COUNT(*) AS reviews_huerfanas
FROM reviews r
LEFT JOIN listings l
    ON r.listing_id = l.id
WHERE l.id IS NULL;
"""
print(pd.read_sql(query, con))

   reviews_huerfanas
0               6592


**Interpretación**: 6.592 reseñas (0,64% del total) tienen un `listing_id` que no existe en el snapshot actual de `listings`. Una posible explicación es que esos anuncios hayan sido dados de baja después de registrarse las reseñas, aunque el dataset por sí solo no permite confirmar la causa. Es un **riesgo de integridad referencial**: estas reseñas no pueden asociarse de forma fiable con atributos del alojamiento presentes en `listings` (por ejemplo, distrito), por lo que deben tratarse antes de cualquier análisis agregado que dependa de esos atributos.

## 4. Autenticidad: patrones sospechosos

**Criterio**: no afirmamos que una reseña sea falsa, eso requeriría acceso a los sistemas internos de Airbnb. Documentamos **patrones estadísticos** que un equipo de confianza y seguridad investigaría:
- Mismo usuario (`reviewer_id`) reutilizando exactamente el mismo texto en alojamientos distintos (`listing_id` distinto): un patrón compatible con texto reutilizado (plantilla personal o frase genérica), no necesariamente con reseñas falsas.
- Reseñas con menos de 5 caracteres: umbral operativo para identificar comentarios extremadamente breves. La longitud por sí sola no implica que el comentario sea inválido o falso, por ejemplo "Good" o "Bien" sí expresan una valoración, pero estos comentarios aportan muy poca información textual para analizar en profundidad la experiencia descrita.

In [6]:
rev_texto = reviews.dropna(subset=["comments"]).copy()

grupos_reviewer_texto = rev_texto.groupby(["reviewer_id", "comments"])["listing_id"].nunique()
grupos_multi_alojamiento = grupos_reviewer_texto[grupos_reviewer_texto > 1]
print("Combinaciones de reviewer_id + texto que aparecen en 2 o mas alojamientos distintos:", len(grupos_multi_alojamiento))

conteo_por_grupo = rev_texto.groupby(["reviewer_id", "comments"]).size()
apariciones_repetidas_multi_alojamiento = (conteo_por_grupo[grupos_reviewer_texto > 1] - 1).sum()
print("Apariciones posteriores a la primera, dentro de esos grupos:", apariciones_repetidas_multi_alojamiento)

print()
print("De esas combinaciones, en cuantos alojamientos distintos aparecen:")
print(grupos_multi_alojamiento.value_counts().sort_index())

idx_multi_alojamiento = grupos_multi_alojamiento.index
en_grupo_multi_alojamiento = rev_texto.set_index(["reviewer_id", "comments"]).index.isin(idx_multi_alojamiento)
longitud_texto_reutilizado = rev_texto.loc[en_grupo_multi_alojamiento, "comments"].str.len()
print()
print("Longitud del texto reutilizado, mediana:", longitud_texto_reutilizado.median())
print("Longitud de comments en general, mediana:", rev_texto["comments"].str.len().median())

rev_texto["len_comentario"] = rev_texto["comments"].astype(str).str.len()
print()
print("Comentarios con menos de 5 caracteres:", (rev_texto["len_comentario"] < 5).sum())
print()
print("Ejemplos de comentarios muy cortos:")
print(rev_texto.loc[rev_texto["len_comentario"] < 5, "comments"].value_counts().head(10))

Combinaciones de reviewer_id + texto que aparecen en 2 o mas alojamientos distintos: 452


Apariciones posteriores a la primera, dentro de esos grupos: 512

De esas combinaciones, en cuantos alojamientos distintos aparecen:
listing_id
2    413
3     32
4      5
5      1
6      1
Name: count, dtype: int64



Longitud del texto reutilizado, mediana: 14.0


Longitud de comments en general, mediana: 172.0



Comentarios con menos de 5 caracteres: 9917

Ejemplos de comentarios muy cortos:
comments
.       2719
Good     707
Top      473
-        366
good     353
👍        280
Bien     246
Ok       233
Nice     226
ok       110
Name: count, dtype: int64


**Interpretación**: 452 combinaciones de `reviewer_id` + texto aparecen en 2 o más alojamientos distintos, con 512 apariciones posteriores a la primera dentro de esos grupos (mismo criterio de conteo que se usa en el resto del notebook). (Nota: quedan fuera de esta cifra 91 casos de mismo `reviewer_id` y mismo texto dentro de un único alojamiento, esos no cumplen el criterio de "alojamientos distintos" definido arriba.)

La intensidad de la reutilización es, en la gran mayoría de los casos, baja: el 91,4% de esas 452 combinaciones (413) aparece en exactamente 2 alojamientos, no en muchos más. Además, el texto reutilizado es mucho más corto que un comentario típico (mediana de 14 caracteres, frente a 172 en el conjunto de `comments`), lo que apunta a que la mayor parte de estos casos son frases genéricas cortas ("Great place", ":-)", "Todo perfecto") reutilizadas de forma que podría ser casual, aunque estos datos no permiten determinar la intención. Esto no descarta el patrón como señal a vigilar, pero sí lo matiza: no constituye, en conjunto, evidencia fuerte de manipulación, aunque no se puede excluir que casos individuales dentro de este grupo sí lo sean.

Por separado, 9.917 reseñas tienen menos de 5 caracteres. Esto no implica que sean falsas ni inválidas (algunas sí transmiten una valoración mínima, como "Good", "Bien" o "Top"), pero por su extrema brevedad aportan poca información textual detallada sobre la experiencia. Es un patrón de baja riqueza informativa del contenido, una dimensión distinta de la autenticidad. Ninguno de los dos patrones se elimina en esta fase, se documentan para su priorización en el tratamiento.

## 5. Coherencia: fechas de las reseñas

**Criterio**: el snapshot de este proyecto está fechado el 2026-06-24 (Inside Airbnb). Una reseña con fecha posterior a esa fecha es un **valor sospechoso**, no un valor imposible: no es una contradicción lógica dentro de la propia fila (a diferencia de `minimum_nights` > `maximum_nights` en `calendar`), es una comparación contra una fecha de referencia externa. Podría deberse a que `reviews.csv` se extrajo unos días después que `listings.csv` dentro del mismo paquete de snapshot, no necesariamente a un error.

In [7]:
reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")

fechas_invalidas = reviews["date"].isna().sum()
print("Fechas no interpretables como fecha valida:", fechas_invalidas)

fecha_snapshot = pd.Timestamp("2026-06-24")
posteriores = (reviews["date"] > fecha_snapshot).sum()
print("Reviews con fecha posterior al snapshot (2026-06-24):", posteriores, f"({posteriores/len(reviews)*100:.3f}%)")
print("Fecha minima:", reviews["date"].min(), "| Fecha maxima:", reviews["date"].max())

Fechas no interpretables como fecha valida: 0
Reviews con fecha posterior al snapshot (2026-06-24): 122 (0.012%)
Fecha minima: 2010-10-03 00:00:00 | Fecha maxima: 2026-07-02 00:00:00


**Interpretación**: todas las fechas son interpretables (0 fechas inválidas). El rango va de 2010-10-03 (consistente con el histórico de reseñas de Airbnb) a 2026-07-02, 8 días después de la fecha de snapshot indicada por Inside Airbnb (2026-06-24). 122 reseñas (0,012% del total) tienen fecha posterior al snapshot. Dado el desfase pequeño (8 días) y el volumen mínimo de casos, es compatible con una diferencia de unos días entre la extracción de `reviews.csv` y `listings.csv` dentro del mismo paquete de snapshot, no con un error sistemático de fechas. Se documenta como observación, no se clasifica como hallazgo con umbral objetivo (mismo criterio aplicado a `price` en el notebook de `listings` y a los valores frecuentes de `maximum_nights` en el notebook de `calendar`).

## Resumen de hallazgos — tabla `reviews`

| # | Hallazgo | Dimensión | Cifra |
|---|---|---|---|
| R1 | Comentarios con texto idéntico a otra reseña | Duplicados | 48.302 |
| R2 | Reseñas huérfanas (SQL, `LEFT JOIN` contra `listings`) | Integridad referencial | 6.592 (0,64%) |
| R3 | Mismo `reviewer_id` con texto idéntico en alojamientos distintos | Autenticidad | 512 (en 452 combinaciones) |
| R4 | Reseñas con menos de 5 caracteres | Riqueza informativa | 9.917 |

Nota: R3 es un subconjunto de R1, no un fenómeno aparte que se sume al total. R1 cuenta cualquier texto repetido entre reseñas, sin importar el autor; R3 aísla, dentro de ese mismo fenómeno, los casos donde además coincide el mismo `reviewer_id` en alojamientos distintos, la combinación más relevante para autenticidad.

**Verificaciones con resultado satisfactorio** (se comprobaron y no se detectaron problemas de calidad, por lo que no se incluyen en la tabla de hallazgos):

- Nulos en `comments` (114, 0,011%) y `reviewer_name` (2, 0,0002%): ambos por debajo del umbral de completitud definido en la Sección 1.
- Filas completamente duplicadas: 0. El identificador `id` es único.

**Observación sin criterio objetivo de clasificación** (se documenta para contexto, no se convierte en hallazgo por falta de evidencia de error):

- 122 reseñas (0,012%) con fecha posterior al snapshot de `listings` (2026-06-24), hasta el 2026-07-02. Desfase pequeño y volumen mínimo, compatible con una extracción de `reviews.csv` unos días después de `listings.csv`.

Estos 4 hallazgos pasan a la tabla de recomendaciones de gobernanza, junto con los de `listings` y `calendar`.